# 🎬 Sentiment Analysis — IMDB Reviews
### CV-Level NLP Project
**Dataset:** IMDB Dataset.csv (50,000 reviews) | **Task:** Binary Sentiment Classification (Positive / Negative)

**Pipeline:** Data Collection (CSV) → EDA → Preprocessing → TF-IDF → Model Training → Evaluation → Model Feature WordClouds → Deployment

## 1. 📦 Install & Import Dependencies

In [ ]:
# Install (uncomment if running fresh)
# !pip install pandas numpy nltk scikit-learn matplotlib seaborn wordcloud datasets joblib

import os, re, pickle, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report, confusion_matrix
)
from datasets import load_dataset

warnings.filterwarnings('ignore')
print('✅ Imports done')

In [ ]:
# Download NLTK assets
for pkg in ['stopwords', 'wordnet', 'punkt', 'punkt_tab', 'omw-1.4']:
    nltk.download(pkg, quiet=True)
print('✅ NLTK assets ready')

## 2. 📂 Data Collection — Loading from CSV

In [ ]:
csv_path = 'IMDB Dataset.csv'

# If CSV is missing, safely download it first (for portability)
if not os.path.exists(csv_path):
    print('Downloading raw dataset to CSV...')
    dataset = load_dataset('imdb')
    _df = pd.concat([dataset['train'].to_pandas(), dataset['test'].to_pandas()], ignore_index=True)
    _df.rename(columns={'label': 'sentiment'}, inplace=True)
    _df.to_csv(csv_path, index=False)

print('Loading dataset from CSV...')
df = pd.read_csv(csv_path)

# Handle missing values if any
initial_len = len(df)
df.dropna(inplace=True)
if len(df) < initial_len:
    print(f'Removed {initial_len - len(df)} rows with missing values.')

print(f'Shape       : {df.shape}')
print(f'Total Reviews : {len(df):,}')
print(f'Positive (1): {(df["sentiment"] == 1).sum():,}')
print(f'Negative (0): {(df["sentiment"] == 0).sum():,}')
df.head(3)

## 3. 🔍 EDA — Exploratory Data Analysis

In [ ]:
# Sentiment Distribution
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

counts = df['sentiment'].value_counts().rename({0: 'Negative', 1: 'Positive'})
colors = ['#e74c3c', '#2ecc71']
bars = axes[0].bar(counts.index, counts.values, color=colors, edgecolor='white', width=0.5)
for bar, val in zip(bars, counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
                 f'{val:,}', ha='center', va='bottom', fontsize=11, fontweight='bold')
axes[0].set_title('Sentiment Distribution', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Count')
axes[0].spines[['top','right']].set_visible(False)

# Review length distribution
df['length'] = df['text'].str.split().str.len()
axes[1].hist(df[df['sentiment']==1]['length'], bins=60, alpha=0.6, color='#2ecc71', label='Positive')
axes[1].hist(df[df['sentiment']==0]['length'], bins=60, alpha=0.6, color='#e74c3c', label='Negative')
axes[1].set_title('Review Length Distribution', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Word Count')
axes[1].set_ylabel('Frequency')
axes[1].legend()
axes[1].spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.show()

print(f'Average review length: {df["length"].mean():.0f} words')
print(f'Median review length : {df["length"].median():.0f} words')

In [ ]:
# Sample reviews
print('=== Sample POSITIVE Review ===')
print(df[df['sentiment']==1]['text'].iloc[0][:500])
print('\n=== Sample NEGATIVE Review ===')
print(df[df['sentiment']==0]['text'].iloc[0][:500])

## 4. 🔧 Text Preprocessing

In [ ]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))
custom_stops = {'movie', 'film', 'one', 'character', 'time', 'story', 'scene', 'people', 'watch', 'make', 'even', 'really'}
stop_words.update(custom_stops)

HTML_RE = re.compile(r'<[^>]+>')
URL_RE  = re.compile(r'http\S+|www\S+')
PUNC_RE = re.compile(r'[^a-z\s]')

def preprocess(text: str) -> str:
    text = str(text).lower()
    text = HTML_RE.sub(' ', text)
    text = URL_RE.sub(' ', text)
    text = PUNC_RE.sub(' ', text)
    tokens = word_tokenize(text)
    tokens = [lemmatizer.lemmatize(t) for t in tokens
              if t not in stop_words and len(t) > 2]
    return ' '.join(tokens)

print('Preprocessing 50,000 reviews (may take a few minutes)...')
df['clean_text'] = df['text'].apply(preprocess)
print('✅ Preprocessing complete')
print('\nSample clean text:')
print(df['clean_text'].iloc[0][:300])

## 5. ⚙️ Feature Engineering — TF-IDF Vectorization

In [ ]:
X = df['clean_text']
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {len(X_train):,} reviews  |  Test: {len(X_test):,} reviews')

tfidf = TfidfVectorizer(max_features=50_000, ngram_range=(1, 2), sublinear_tf=True)
X_train_vec = tfidf.fit_transform(X_train)
X_test_vec  = tfidf.transform(X_test)

print(f'Vocabulary size: {len(tfidf.vocabulary_):,}')
print(f'Feature matrix (train): {X_train_vec.shape}')

## 6. 🤖 Model Training — Logistic Regression & Naive Bayes

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, C=5.0, solver='saga', n_jobs=-1),
    'Naive Bayes'        : MultinomialNB(alpha=0.1)
}

results = {}
for name, model in models.items():
    print(f'\nTraining {name}...')
    model.fit(X_train_vec, y_train)
    preds = model.predict(X_test_vec)
    results[name] = {
        'model' : model,
        'preds' : preds,
        'cm'    : confusion_matrix(y_test, preds),
        'acc'   : accuracy_score(y_test, preds),
        'prec'  : precision_score(y_test, preds),
        'rec'   : recall_score(y_test, preds),
        'f1'    : f1_score(y_test, preds),
    }
    r = results[name]
    print(f'  Accuracy : {r["acc"]:.4f}')
    print(f'  Precision: {r["prec"]:.4f}')
    print(f'  Recall   : {r["rec"]:.4f}')
    print(f'  F1-Score : {r["f1"]:.4f}')

## 7. 📊 Evaluation — Metrics & Comparison

In [ ]:
# Classification reports
for name, res in results.items():
    print(f'=== {name} ===')
    print(classification_report(y_test, res['preds'], target_names=['Negative','Positive']))
    print()

In [ ]:
# Model Comparison Bar Chart
metrics = ['acc', 'prec', 'rec', 'f1']
labels  = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
x = np.arange(len(metrics))
width = 0.35
palette = ['#3498db', '#e67e22']

fig, ax = plt.subplots(figsize=(10, 5))
for i, (name, res) in enumerate(results.items()):
    vals = [res[m] for m in metrics]
    bars = ax.bar(x + i*width - width/2, vals, width, label=name, color=palette[i], edgecolor='white')
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
                f'{v:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylim(0.78, 1.02)
ax.set_ylabel('Score')
ax.set_title('Model Comparison — Evaluation Metrics', fontsize=14, fontweight='bold')
ax.legend()
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
# Confusion Matrix Heatmaps
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, (name, res) in zip(axes, results.items()):
    sns.heatmap(res['cm'], annot=True, fmt='d', cmap='Blues',
                xticklabels=['Negative','Positive'],
                yticklabels=['Negative','Positive'],
                linewidths=0.5, linecolor='white', ax=ax)
    ax.set_title(f'Confusion Matrix\n{name}', fontsize=12, fontweight='bold')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
plt.tight_layout()
plt.show()

## 8. ☁️ Feature-Driven WordClouds (Using Model Coefficients)

In [ ]:
feature_names = tfidf.get_feature_names_out()
lr_model = results['Logistic Regression']['model']
coefs = lr_model.coef_[0]

# Top 100 Positive & Negative Features
pos_idx = {feature_names[i]: coefs[i] for i in range(len(feature_names)) if coefs[i] > 0}
top_pos = dict(sorted(pos_idx.items(), key=lambda x: x[1], reverse=True)[:100])

neg_idx = {feature_names[i]: -coefs[i] for i in range(len(feature_names)) if coefs[i] < 0}
top_neg = dict(sorted(neg_idx.items(), key=lambda x: x[1], reverse=True)[:100])

def plot_coef_wordcloud(freq_dict, bg_color, cmap, title):
    wc = WordCloud(width=800, height=400, background_color=bg_color, colormap=cmap, max_words=100)
    wc.generate_from_frequencies(freq_dict)
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.imshow(wc, interpolation='bilinear')
    ax.axis('off')
    fig.patch.set_facecolor(bg_color)
    t_color = 'black' if bg_color == 'white' else 'white'
    ax.set_title(title, fontsize=16, fontweight='bold', pad=14, color=t_color)
    plt.show()

plot_coef_wordcloud(top_pos, 'white', 'Greens', 'Top Positive Sentiment Drivers (LR Coefficients)')
plot_coef_wordcloud(top_neg, '#0a0a0a', 'Reds', 'Top Negative Sentiment Drivers (LR Coefficients)')


## 9. 💾 Save Model & Vectorizer

In [ ]:
os.makedirs('models', exist_ok=True)

# Save TF-IDF vectorizer
with open('models/vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf, f)

# Find & save best model
best_name  = max(results, key=lambda n: results[n]['f1'])
best_model = results[best_name]['model']
with open('models/model.pkl', 'wb') as f:
    pickle.dump(best_model, f)

print(f'✅ Best Model  : {best_name}')
print(f'   Saved → models/model.pkl  &  models/vectorizer.pkl')

## 10. 🚀 Deploy
Run the Streamlit app:
```bash
streamlit run app.py
```
Open **http://localhost:8501** — enter any review text → get Positive / Negative prediction with confidence scores.